# COMPETICION.ipynb — Entregable final del equipo

**Equipo**: Oscar · Dani · Fernando

**Flujo de trabajo**:
1. Cada miembro ejecuta su `exp_oscar.ipynb` / `exp_dani.ipynb` / `exp_fernando.ipynb` completo.
2. Cada uno exporta su mejor modelo a `models/<nombre>.keras` y `results/<nombre>.json`.
3. Este notebook carga los 3 modelos, verifica compatibilidad y construye el ensemble.

---
⚠️ **GPU workaround obligatorio** (RTX 5070 Ti Blackwell incompatible con TF GPU):  
ejecutar la celda siguiente **antes** de cualquier import de TF/Keras.

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'   # CPU-only — ANTES de cualquier import TF

import sys, json
sys.path.insert(0, os.getcwd())
import numpy as np
import pandas as pd
import warnings
warnings.simplefilter('ignore')

from utils import (
    create_time_series_data, make_splits, load_data,
    eval_mae_naive,
    FILEPATH_SHARED, V_IN_SHARED, V_OUT_SHARED,
    FFD_D_SHARED, PRICE_COLS_SHARED, RETURN_COLS_SHARED,
)
print('OK — modo CPU activo')

## ☑️ Checklist del enunciado — RELLENAR A MANO antes de ejecutar nada

**Leer el enunciado completo y contestar antes de tocar el código.**

### Qué hay que predecir
- [ ] ¿Regresión (valor continuo) o clasificación (categoría)?  → `TIPO_PROBLEMA = ___`
- [ ] ¿Cuántos días de horizonte de predicción?               → `V_OUT_SHARED` en `utils.py` = ___
- [ ] ¿Cuántos días de historia de entrada?                   → `V_IN_SHARED` en `utils.py` = ___
- [ ] ¿Predecir un activo o varios?                           → univariante / multivariante

### Métrica y entregable
- [ ] ¿Qué métrica evalúa el hackathon?   → `METRICA = ___`  (MAE / RMSE / accuracy / otra)
- [ ] ¿Qué hay que entregar exactamente?  → CSV / MAE en test / notebook / otro: ___
- [ ] ¿Hay columna target explícita o hay que derivarla?

### Datos
- [ ] ¿El CSV trae precios o retornos?    → `PRICE_COLS_SHARED` / `RETURN_COLS_SHARED` en `utils.py`
- [ ] ¿Índice temporal?                  → `shuffle=False` obligatorio si lo es

### Pre-vuelo
- [ ] Los 3 miembros han ejecutado su `exp_*.ipynb` completo hasta la sección 7
- [ ] Existen `models/oscar.keras`, `models/dani.keras`, `models/fernando.keras`
- [ ] Existen `results/oscar.json`, `results/dani.json`, `results/fernando.json`

## 0. Cargar datos — usando las constantes compartidas del equipo

Mismo split que usaron los 3 notebooks personales (mismas constantes, misma semilla).

In [ ]:
FILEPATH = FILEPATH_SHARED
V_IN     = V_IN_SHARED
V_OUT    = V_OUT_SHARED
FFD_D    = FFD_D_SHARED

X_src, y_src = load_data(FILEPATH, price_cols=PRICE_COLS_SHARED,
                          return_cols=RETURN_COLS_SHARED, ffd_d=FFD_D)

X, _  = create_time_series_data(X_src, V_IN, V_OUT)
_, y  = create_time_series_data(y_src, V_IN, V_OUT)
X_tr, X_v, X_ts, y_tr, y_v, y_ts = make_splits(X, y)

N_FEAT    = X.shape[2]
N_TARGETS = y.shape[1]

print(f'X: {X.shape}   y: {y.shape}')
print(f'Train: {X_tr.shape[0]:,}  Val: {X_v.shape[0]:,}  Test: {X_ts.shape[0]:,}')
print(f'n_features={N_FEAT}  n_targets={N_TARGETS}')

naive_mae = eval_mae_naive(X_ts, y_ts)
print(f'\nBaseline naive: MAE = {naive_mae:.4f}')

## 1. Cargar los 3 modelos y sus resultados

In [ ]:
import tensorflow as tf

MIEMBROS  = ['oscar', 'dani', 'fernando']
modelos   = {}
metas     = {}
faltantes = []

for nombre in MIEMBROS:
    model_path = f'models/{nombre}.keras'
    json_path  = f'results/{nombre}.json'

    ok_model = os.path.exists(model_path)
    ok_json  = os.path.exists(json_path)

    if not ok_model:
        print(f'[FALTA] {nombre}: modelo no encontrado en {model_path}')
        faltantes.append(nombre)
        continue
    if not ok_json:
        print(f'[FALTA] {nombre}: JSON no encontrado en {json_path}')
        faltantes.append(nombre)
        continue

    modelos[nombre] = tf.keras.models.load_model(model_path)
    with open(json_path) as f:
        metas[nombre] = json.load(f)
    print(f'[OK]    {nombre}: {metas[nombre]["modelo"]}  '
          f'val={metas[nombre]["mae_val"]:.4f}  '
          f'test={metas[nombre]["mae_test"]:.4f}')

print()
if faltantes:
    print(f'AVISO: faltan modelos de: {faltantes}')
    print('  Cada miembro debe ejecutar su exp_*.ipynb hasta la seccion 7 antes de continuar.')
else:
    print('Los 3 modelos cargados correctamente.')

## 2. Verificar compatibilidad — shapes deben coincidir

Para que el ensemble sea válido, los 3 modelos deben tener el mismo input shape
y output shape. Si los shapes no coinciden el ensemble no es posible y se muestra
un mensaje explicativo con la causa probable y la solución.

In [ ]:
ENSEMBLE_OK = False

if len(modelos) < len(MIEMBROS):
    print('No se puede verificar: faltan modelos. Ver seccion anterior.')
else:
    shapes_input  = {n: m.input_shape  for n, m in modelos.items()}
    shapes_output = {n: m.output_shape for n, m in modelos.items()}

    print('Input shapes:')
    for n, s in shapes_input.items():
        print(f'  {n}: {s}')

    print('\nOutput shapes:')
    for n, s in shapes_output.items():
        print(f'  {n}: {s}')

    inputs_ok  = len({str(s) for s in shapes_input.values()})  == 1
    outputs_ok = len({str(s) for s in shapes_output.values()}) == 1

    if not inputs_ok:
        print('\n[ERROR] Los modelos tienen INPUT shapes distintos.')
        print('  Causa probable: distintos V_IN o n_features.')
        print('  Solucion: verificar que los 3 usan FILEPATH_SHARED/V_IN_SHARED de utils.py.')
        print('  El ensemble automatico NO puede construirse. Usar la seccion de fallback.')
    elif not outputs_ok:
        print('\n[ERROR] Los modelos tienen OUTPUT shapes distintos.')
        print('  Causa probable: distintos n_targets (V_OUT o numero de activos distinto).')
        print('  Solucion: verificar que V_OUT_SHARED es el mismo para los 3.')
        print('  El ensemble automatico NO puede construirse. Usar la seccion de fallback.')
    else:
        print('\n[OK] Shapes compatibles — el ensemble es valido.')
        ENSEMBLE_OK = True

## 3. Tabla comparativa de los 3 modelos

In [ ]:
if metas:
    rows = []
    for nombre, meta in metas.items():
        rows.append({
            'miembro':   nombre,
            'modelo':    meta['modelo'],
            'V_IN':      meta['V_IN'],
            'V_OUT':     meta['V_OUT'],
            'epochs':    meta['epochs'],
            'mae_val':   meta['mae_val'],
            'mae_test':  meta['mae_test'],
            'dir_acc':   meta['dir_acc'],
            'params':    meta['params'],
            'ens_test':  meta.get('mae_ensemble_test', float('nan')),
        })
    df_comp = pd.DataFrame(rows).set_index('miembro').sort_values('mae_test')
    display(df_comp)

    mejor_individual = df_comp['mae_test'].idxmin()
    print(f'\nMejor modelo individual: {mejor_individual}  '
          f'MAE test = {df_comp.loc[mejor_individual, "mae_test"]:.4f}')
    print(f'Naive baseline:          MAE test = {naive_mae:.4f}')
else:
    print('No hay metas cargadas. Ver seccion 1.')

## 4. Ensemble automático de los 3 modelos

Promedia las predicciones de los 3 modelos sobre el test set.  
Cada modelo aporta un enfoque diferente → la diversidad entre miembros  
reduce la varianza del error conjunto.

In [ ]:
if not ENSEMBLE_OK:
    print('Ensemble desactivado por incompatibilidad de shapes. Ver seccion 2.')
elif len(modelos) < len(MIEMBROS):
    print('Ensemble desactivado: faltan modelos. Ver seccion 1.')
else:
    preds = []
    print('Predicciones individuales en test:')
    for nombre, model in modelos.items():
        pred    = model.predict(X_ts, verbose=0)
        mae_ind = float(np.mean(np.abs(pred - y_ts)))
        preds.append(pred)
        print(f'  {nombre}: MAE = {mae_ind:.4f}')

    y_ens   = np.mean(preds, axis=0)
    mae_ens = float(np.mean(np.abs(y_ens - y_ts)))

    mae_mejor_ind = df_comp['mae_test'].min()
    mejora_vs_ind   = (mae_ens - mae_mejor_ind) / mae_mejor_ind
    mejora_vs_naive = (mae_ens - naive_mae) / naive_mae

    print(f'\n== RESULTADO FINAL ==')
    print(f'  Naive baseline:       MAE = {naive_mae:.4f}')
    print(f'  Mejor individual:     MAE = {mae_mejor_ind:.4f}  ({mejor_individual})')
    print(f'  Ensemble 3 modelos:   MAE = {mae_ens:.4f}')
    print(f'  vs mejor individual:  {mejora_vs_ind:+.1%}')
    print(f'  vs naive:             {mejora_vs_naive:+.1%}')
    print(f'\n-> MAE ENTREGABLE: {mae_ens:.4f}')
    RESULTADO_FINAL = mae_ens

## 5. Fallback manual — si el ensemble automático falla

Usar esta sección si la sección 4 falla por modelos incompatibles, ficheros
no encontrados u otro problema. El entregable **nunca depende al 100%** del
ensemble automático.

**Pasos**:
1. Ver la tabla de sección 3 para identificar el mejor config individual.
2. Rellenar `TIPO_FALLBACK`, `LR_FALLBACK`, `N_SEEDS_FALLBACK`.
3. Descomentar y ejecutar la celda siguiente.
4. Usar `ens_fallback['mae_test']` como resultado entregable.

In [ ]:
# ── FALLBACK MANUAL — descomentar y ajustar si el ensemble automatico falla ──

# # 1. Config del mejor modelo individual (copiar de la tabla de seccion 3)
# TIPO_FALLBACK     = 'lstm'   # <- ajustar segun tabla
# LR_FALLBACK       = 3e-4    # <- ajustar segun tabla
# N_SEEDS_FALLBACK  = 5
# EPOCHS_FALLBACK   = 200
#
# # 2. Importar lo que falta (si la seccion 0 no se ejecuto)
# # from utils import build_model, train_model, train_ensemble, evaluate
#
# # 3. Entrenar ensemble del mejor config
# from utils import train_ensemble, build_model
# ens_fallback = train_ensemble(
#     TIPO_FALLBACK, X_tr, y_tr, X_v, y_v, X_ts, y_ts,
#     V_in=V_IN, n_features=N_FEAT, n_targets=N_TARGETS,
#     n_seeds=N_SEEDS_FALLBACK, epochs=EPOCHS_FALLBACK, lr=LR_FALLBACK,
# )
# print(f'Fallback MAE val:  {ens_fallback["mae_val"]:.4f}')
# print(f'Fallback MAE test: {ens_fallback["mae_test"]:.4f}')
# RESULTADO_FINAL = ens_fallback['mae_test']
# print(f'-> MAE ENTREGABLE (fallback): {RESULTADO_FINAL:.4f}')

print('Seccion de fallback lista para usar si es necesario.')
print('Descomentar el bloque de codigo de arriba y ajustar los parametros.')